# Lecture14 LLM時代のデータサイエンスとは

【2026年度版】
- Transformer v5(Colabのデフォルトバージョン)にあわせて修正


【2024年度版】

ChatGPTや大規模言語モデル(LLM)の元となったTransformerの概要やHuggingFaceを利用した実行例を試し、最後に、データサイエンスとの関係性についていくつか話題を実例を含めながら紹介する。

- 深層学習から発展し、ChatGPTなどのもとになった、Transformerについて、動かしてみる。（時系列データ予測もやってみる）
- LLMの活用例
 - CSVデータエージェント
 - Q&A with sources

【注意】実行環境は様々なライブラリのバージョンに依存します、将来の動作は全く保証されません。

## Transformer

ここでは「機械学習エンジニアのためのTransfomers」の例を実際に動かしてみます。

ここでは、HuggingFaceというオープンソースのモデルやデータをあつめたハブからpipeline()関数を使ってローカルにtransformerのモデルをダウンロード（実際にはcolabの仮想マシンに）して使っています。


https://github.com/nlp-with-transformers/notebooks/blob/main/01_introduction.ipynb

## Transformerの歴史
Transformerとは、深層学習のアーキテクチャの一つであり、Googleにより発明されました。それ以降、BERTやGPTなどが開発され、現在のGPT3.5やGPT4すなわちChatGPTのもとになるTransformerが開発されてきました。

![Transformerの歴史](https://raw.githubusercontent.com/nlp-with-transformers/notebooks/48e4a5e5c44b86e1593c0945a49af9675cfd7158//images/chapter01_timeline.png)

## 「エンコーダー・デコーダ」アーキテクチャ

深層学習の中で、RNN(再帰型ニューラルネットワーク）を用いて、入力文字の順序を持つ列を、状態にエンコードする、エンコーダーブロックと、状態からこれを、出力文字列に、やはりRNNを用いてでコードするデコードブロックを組み合わせたものが、トランスフォーマの基本となるアーキテクチャです。
![](https://raw.githubusercontent.com/nlp-with-transformers/notebooks/48e4a5e5c44b86e1593c0945a49af9675cfd7158//images/chapter01_enc-dec.png)

In [ ]:
!pip install diffusers accelerate scipy safetensors sacremoses

例題とするテキスト（英語）です、~~Amazon~~EC業者への問い合わせのメールのようです。

In [ ]:
text = """Dear Amazon, last week I ordered an Optimus Prime action figure \
from your online store in Germany. Unfortunately, when I opened the package, \
I discovered to my horror that I had been sent an action figure of Megatron \
instead! As a lifelong enemy of the Decepticons, I hope you can understand my \
dilemma. To resolve the issue, I demand an exchange of Megatron for the \
Optimus Prime figure I ordered. Enclosed are copies of my records concerning \
this purchase. I expect to hear from you soon. Sincerely, Bumblebee."""

## テキスト分類の例
pipleline関数に、タスク種別を引数に指定すると、適切なモデルを自動選択して、モデルのインスタンスをcolab仮想マシンに生成します。（具体的なモデルを指定することもできます）。

In [ ]:
from transformers import pipeline
#distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english)

classifier = pipeline("text-classification")

具体的に選択されたモデルを表示します。

In [ ]:
classifier.model.name_or_path

In [ ]:
classifier.model

このようにして、与えた文章に対して感情を分析してくれます。スコア付き。

In [ ]:
classifier(text)

一度、モデルを作れば、あとは何度も別のテキストを試せます。ディープラーニングの父、ヒントン教授のインタビューから

In [ ]:
text_hinton="""Geoff Hinton

Yes, I do. I strongly believe that. I strongly believe that when we eventually understand how the brain works,
 that's going to give us lots of psychological insight too. Just as understanding chemistry at the atomic level,
 understanding of how molecules bump into each other and what happens,
 gives us lots of insight into the gas laws.
"""

In [ ]:
classifier(text_hinton)

## 固有語の識別

named entity(名前のあるエンティティ)の識別

In [ ]:
import pandas as pd
#dbmdz/bert-large-cased-finetuned-conll03-english and revision f2482bf

ner_tagger = pipeline("ner", aggregation_strategy="simple")

outputs = ner_tagger(text)
pd.DataFrame(outputs)

## 質問に答える

In [ ]:

reader = pipeline("question-answering")
question = "What does the customer want?"
outputs = reader(question=question, context=text)
pd.DataFrame([outputs])

## 要約をする

### 2026年 transformer v5でぎりぎり動いた

In [ ]:

import torch
from transformers import pipeline

# Any other chat model will also work - if you're low on memory you can use a smaller one
summarizer = pipeline("text-generation", model="Qwen/Qwen3-4B-Instruct-2507")
message_history = [
    {
        "role": "user",
        "content": "Summarize the following text:\n\n"+text
    }
]
print(summarizer(message_history)[0]["generated_text"][-1]["content"])

## 翻訳では、~~FuguMTを使ってみます。~~

'text=generatoin'を使います(2026)


In [ ]:
from transformers import pipeline
#translator = pipeline("text-generation", model="Qwen/Qwen3-4B-Instruct-2507")
messages = [{"role": "user", "content": "Translate this into Japanese:\n\nHello, how are you?"}]

out = summarizer(messages)
print(out[0]["generated_text"][-1]["content"])

In [ ]:
#例(高慢と偏見 Jane Austen)
txt="""It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.
 However little known the feelings or views of such a man may be on his first entering a neighbourhood, this truth is so well
  fixed in the minds of the surrounding families, that he is considered the rightful property of some one or other of their daughters.
"""
messages = [{"role": "user", "content": "Translate this into Japanese:\n\n"+txt}]

out = summarizer(messages)
print(out[0]["generated_text"][-1]["content"])

## 画像生成
テキストから画像を生成する有名な、stablefusionも試せます。

In [ ]:
!pip install diffusers transformers accelerate scipy safetensors

メモリ利用量が無料版のcolabを超えるかもしれません。その場合は、クラッシュします。がメモリがリセットされるので再実行します。

In [ ]:
from diffusers import StableDiffusionPipeline, EulerDiscreteScheduler
import torch

model_id = "stabilityai/stable-diffusion-2"
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

scheduler = EulerDiscreteScheduler.from_pretrained(model_id, subfolder="scheduler")
pipe = StableDiffusionPipeline.from_pretrained(model_id, scheduler=scheduler, torch_dtype=torch.float16)
pipe = pipe.to("cuda")

prompt = "The professor in panda outfits is teaching in a busy and packed class room. Some student is throwing spears to him"
image = pipe(prompt).images[0]
image

In [ ]:
prompt = "The professor in panda outfits is teaching in a busy and packed class room. Some student is throwing spears to him"
image = pipe(prompt).images[0]
image

### OpenAIのapiのkeyを持っている場合は。OpenAIの計算リソースが利用できる

OpenAIの（超高速な）サーバーにあるモデルを使うことができます。ローカルメモリを消費しない。

In [ ]:
!pip install openai  langchain_openai

In [ ]:
# get a token: https://platform.openai.com/account/api-keys

from getpass import getpass

OPENAI_API_KEY = getpass()

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [ ]:
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_openai import OpenAI

template = """Question: {question}

Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=["question"])

llm = OpenAI()

llm_chain = LLMChain(prompt=prompt, llm=llm)

question = "What is electroencephalography?"
answer = llm_chain.run(question)
display(answer)

In [ ]:
fugu_translator(answer)

## 有料のOpenAIのapi_keyを持っていなくても、無料のHugging Faceの計算リソースを使うことができます。

こちらも、HugginfFaceのもつ（たぶん、超高速の）サーバーを利用できます。

In [ ]:
!pip install -U from langchain-huggingface

In [ ]:
# get a token: https://huggingface.co/docs/api-inference/quicktour#get-your-api-token

from getpass import getpass

HUGGINGFACEHUB_API_TOKEN = getpass()

In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HUGGINGFACEHUB_API_TOKEN

In [ ]:
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFaceEndpoint

template = """Question: {question}

Answer: Let's think step by step."""
prompt = PromptTemplate(template=template, input_variables=["question"])

repo_id = "mistralai/Mistral-7B-Instruct-v0.2"
llm = HuggingFaceEndpoint(repo_id=repo_id , temperature=0.5,  max_length=64)

llm_chain = LLMChain(prompt=prompt, llm=llm)

question = "What is electroencephalography?"
answer = llm_chain.run(question)
display(answer)

In [ ]:
fugu_translator(answer)